In [ ]:





#PATH_GSEmtx_pkl =  Enter the path of 'imputed_training_set.pkl' here






# Data Discretization, Formating (to BIMBAM) and EWAS by GEMMA

OZL 20230804. 20250616 update.

Organize GSEmtx (imputed_data) into a series of BIMBAM format files readable by GEMMA: Mean Genotype File (geno.txt), Phenotype File (pheno.txt), and SNP Annotation File (anno.txt, optional).

## Introduction to File Formats

### GSEmtx (imputed_dataset)

Each row represents a sample, and the columns include annotation information (beginning with an exclamation mark) as well as the beta values of 669k CpG sites. The annotation columns include the following:
```
!Sample_geo_accession
!Sample_source_name_ch1
!Sample_platform_id
!Sample_data_processing
!phenotype
!Sex
!age
```

---

The following are BIMBAM format series files; for more details, refer to the bimbam-manual.

### BIMBAM Mean Genotype File

Each row represents a CpG site, the first column is the CpG site name, and the next two columns are the alleles used in SNP studies, with the first listed being the minor allele (minor allele: the less common allele for a SNP). These have no significance in this methylation study and are temporarily set as 'G, C'. The subsequent columns contain the data values for each sample at that site. In SNP studies, the data values represent the dosage of the minor allele. Therefore, in this study, G can be considered as "methylated" and C as "unmethylated".

Column names are not specified, and missing values are denoted as 'NA'. An example with three samples and two sites is as follows:

```
cg1, G, C, 0, 2, 1
cg2, G, C, 2, 0, 1
```

### BIMBAM Phenotype File

This file contains phenotype information, namely age. Each row corresponds to the age value of a sample. The order and number of rows (samples) must match the order and number of sample columns in the BIMBAM Mean Genotype File. Row and column names are not set, non-numeric characters are not allowed, and missing values are marked as 'NA'. An example of age information for five samples is as follows:

```
1.2
21
38
79
63
```

### BIMBAM Annotation File (optional)

This file contains location information for CpG probes. The first column is the CpG probe ID, the second column is the base pair position on the chromosome, and the third column is the chromosome number. The order of rows (CpG probes) does not need to match the order in the Mean Genotype File but must include all probes present in the Mean Genotype File. An example is as follows:

```
cg1, 1200, 1
cg2, 1000, 1
cg3, 3320, 1
cg4, 5430, 1
```

## Main

### Setting Environment and Key Parameters

In [1]:
import pandas as pd
import pickle

### File Loading and Checking

In [5]:
GSEmtx = pd.read_pickle(PATH_GSEmtx_pkl)
GSEmtx

,!dataset_id,!trans_age,!sample_id,!sample_source,!age,!sex,!phenotype,cg20789016,cg25085142,cg23416002,...,cg22746501,cg13397747,cg24301352,cg23583742,cg19277269,cg17623046,cg20713761,cg20697010,cg26023912,cg09068759
201503670196_R03C01,GSE147740,1.619048,GSM4444307,blood samples,54.0,F,normal,0.848849,0.539572,0.629344,...,0.744967,0.049448,0.523465,0.854998,0.766016,0.572964,0.620166,0.304182,0.369256,0.489421
201496850202_R05C01,GSE147740,1.238095,GSM4444174,blood samples,46.0,F,normal,0.919322,0.627227,0.656385,...,0.744350,0.044912,0.488530,0.889826,0.750071,0.553639,0.640107,0.255564,0.428855,0.490098
201496850037_R02C01,GSE147740,1.238095,GSM4444085,blood samples,46.0,F,normal,0.873504,0.602947,0.647127,...,0.775070,0.042041,0.447530,0.812539,0.733703,0.571885,0.590251,0.289998,0.433633,0.553147
201496850056_R08C01,GSE147740,1.000000,GSM4444114,blood samples,41.0,F,normal,0.861933,0.749613,0.599099,...,0.778561,0.054613,0.445111,0.924821,0.730846,0.536724,0.591015,0.300360,0.442522,0.498277
201516350011_R01C01,GSE147740,1.380952,GSM4444610,blood samples,49.0,F,normal,0.801570,0.690659,0.629153,...,0.701381,0.050708,0.472226,0.838870,0.746540,0.613943,0.590327,0.195794,0.265195,0.524737
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM5701503,GSE189426,1.428571,GSM5701503,Peripheral blood monocytes,50.0,F,normal,0.914210,0.868764,0.864693,...,0.890535,0.083198,0.848038,0.944484,0.916470,0.910334,0.867148,0.444670,0.706030,0.746051
GSM5701550,GSE189426,1.666667,GSM5701550,Peripheral blood monocytes,55.0,F,normal,0.919991,0.851967,0.865294,...,0.894355,0.086481,0.871683,0.952182,0.909088,0.926590,0.863740,0.475342,0.771335,0.749528
GSM5701551,GSE189426,1.714286,GSM5701551,Peripheral blood monocytes,56.0,F,normal,0.925957,0.811589,0.867579,...,0.914789,0.080551,0.847521,0.948819,0.908238,0.928853,0.863083,0.418450,0.619065,0.720385
GSM5701552,GSE189426,1.857143,GSM5701552,Peripheral blood monocytes,59.0,F,normal,0.921655,0.873528,0.894277,...,0.917316,0.068115,0.843216,0.940922,0.905854,0.912251,0.879810,0.319576,0.528618,0.735804


In [6]:
# Check for missing and abnormal age data:
print('Check for missing and abnormal age data:')
print(GSEmtx.loc[GSEmtx['!age'].astype('str').str.extract(r'([^\d\.]+)').dropna().index]['!age'],'\n')

# View platform statistics
#print('Sample-platforms in this dataset:')
#print(set(GSEmtx['!Sample_platform_id']),'\n')

# Statistics of Missing Methylation Data Values
df_null_list = GSEmtx[GSEmtx.columns[~GSEmtx.columns.str.contains('!')]].isnull()
probe_null_counts = df_null_list.sum(axis=0)
sample_null_counts = df_null_list.sum(axis=1)
all_null_counts = df_null_list.sum().sum()
print('Counts of features have null:\n',probe_null_counts[probe_null_counts>0],sep='')
print('Counts of samples have null\n',sample_null_counts[sample_null_counts>0],sep='')
print('Counts of all null:',all_null_counts,sep='')


Check for missing and abnormal age data:
200992290039_R08C01   NaN
200992310015_R02C01   NaN
Name: !age, dtype: float64 

Counts of features have null:
Series([], dtype: int64)
Counts of samples have null
Series([], dtype: int64)
Counts of all null:0


### Delete samples with empty age

In [7]:
GSEmtx.loc[['200992310015_R02C01','200992290039_R08C01']]

,!dataset_id,!trans_age,!sample_id,!sample_source,!age,!sex,!phenotype,cg20789016,cg25085142,cg23416002,...,cg22746501,cg13397747,cg24301352,cg23583742,cg19277269,cg17623046,cg20713761,cg20697010,cg26023912,cg09068759
200992310015_R02C01,GSE152026,NaN,GSM4599510,Whole Blood,NaN,F,Control,0.850879,0.696126,0.747368,...,0.847967,0.075047,0.667550,0.843207,0.862311,0.687647,0.709969,0.411731,0.526618,0.625662
200992290039_R08C01,GSE152026,NaN,GSM4599453,Whole Blood,NaN,M,Control,0.866355,0.616101,0.666551,...,0.886030,0.070951,0.642859,0.881189,0.805195,0.643327,0.716274,0.428201,0.624142,0.649076


In [8]:
GSEmtx.drop(index=['200992310015_R02C01','200992290039_R08C01'],inplace=True)


In [9]:
print(GSEmtx.shape)
print(GSEmtx['!age'].dtypes)
GSEmtx['!age'].isnull().sum()

(3623, 669281)
float64


0

In [10]:
# Age missing or non-numeric detection
print('Check for missing and abnormal age data:')
print(GSEmtx.loc[GSEmtx['!age'].astype('str').str.extract(r'([^\d\.]+)').dropna().index]['!age'],'\n')
print(GSEmtx['!age'].dtypes)

# View platform statistics
#print('Sample-platforms in this dataset:')
#print(set(GSEmtx['!Sample_platform_id']),'\n')

# Statistics of Missing Methylation Data Values
df_null_list = GSEmtx[GSEmtx.columns[~GSEmtx.columns.str.contains('!')]].isnull()
probe_null_counts = df_null_list.sum(axis=0)
sample_null_counts = df_null_list.sum(axis=1)
all_null_counts = df_null_list.sum().sum()
print('Counts of features have null:\n',probe_null_counts[probe_null_counts>0],sep='')
print('Counts of samples have null\n',sample_null_counts[sample_null_counts>0],sep='')
print('Counts of all null:',all_null_counts,sep='')

Check for missing and abnormal age data:
Series([], Name: !age, dtype: float64) 

float64
Counts of features have null:
Series([], dtype: int64)
Counts of samples have null
Series([], dtype: int64)
Counts of all null:0


## Discretization and Generating BIMBAM Mean Genotype File

get "geno.txt"

In [10]:
temp_geno = GSEmtx[GSEmtx.columns[~GSEmtx.columns.str.contains('!')]].T
temp_geno

,201503670196_R03C01,201496850202_R05C01,201496850037_R02C01,201496850056_R08C01,201516350011_R01C01,201516330018_R06C01,201516330021_R01C01,201530480016_R02C01,201533480011_R04C01,201516330036_R06C01,...,GSM5701498,GSM5701499,GSM5701500,GSM5701501,GSM5701502,GSM5701503,GSM5701550,GSM5701551,GSM5701552,GSM5701553
cg20789016,0.848849,0.919322,0.873504,0.861933,0.801570,0.853478,0.854379,0.864806,0.827355,0.814893,...,0.923870,0.925040,0.921030,0.918246,0.918109,0.914210,0.919991,0.925957,0.921655,0.916299
cg25085142,0.539572,0.627227,0.602947,0.749613,0.690659,0.678936,0.625260,0.664579,0.684338,0.677099,...,0.874246,0.850561,0.865551,0.871102,0.865209,0.868764,0.851967,0.811589,0.873528,0.870246
cg23416002,0.629344,0.656385,0.647127,0.599099,0.629153,0.675085,0.601263,0.741167,0.630401,0.642500,...,0.842372,0.863443,0.874900,0.875796,0.824759,0.864693,0.865294,0.867579,0.894277,0.867790
cg22575874,0.534022,0.465353,0.567584,0.602217,0.648087,0.563714,0.487359,0.578580,0.500575,0.565374,...,0.864009,0.841704,0.824680,0.826570,0.833683,0.856416,0.861029,0.824675,0.847141,0.885733
cg03207983,0.052690,0.047928,0.050383,0.047774,0.046563,0.050578,0.058673,0.054377,0.048243,0.053350,...,0.028597,0.031668,0.026201,0.029731,0.026021,0.025930,0.026151,0.029742,0.030511,0.027739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cg17623046,0.572964,0.553639,0.571885,0.536724,0.613943,0.561284,0.575450,0.630656,0.563102,0.546272,...,0.919721,0.915441,0.924110,0.892690,0.925406,0.910334,0.926590,0.928853,0.912251,0.937627
cg20713761,0.620166,0.640107,0.590251,0.591015,0.590327,0.571205,0.675115,0.602004,0.601684,0.633590,...,0.834807,0.860543,0.836162,0.804176,0.862375,0.867148,0.863740,0.863083,0.879810,0.819769
cg20697010,0.304182,0.255564,0.289998,0.300360,0.195794,0.318657,0.276788,0.266873,0.288517,0.289293,...,0.499765,0.474749,0.458359,0.554525,0.446675,0.444670,0.475342,0.418450,0.319576,0.451124
cg26023912,0.369256,0.428855,0.433633,0.442522,0.265195,0.448273,0.511750,0.586784,0.419862,0.373289,...,0.741628,0.638740,0.678446,0.645180,0.654155,0.706030,0.771335,0.619065,0.528618,0.660774


### Continuous

In [ ]:
geno_2x = 2 * temp_geno

geno_2x.insert(0,'!alle_1','G')
geno_2x.insert(1,'!alle_2','C')
geno_2x.to_csv('geno_2x.txt',header=False)

### d2575 discretization

In [ ]:
geno_d25 = (temp_geno + 0.75).astype('int32')
geno_d75 = (temp_geno + 0.25).astype('int32')

geno_d2575 = geno_d25 + geno_d75

geno_d2575.insert(0,'!alle_1','G')
geno_d2575.insert(1,'!alle_2','C')
geno_d2575.to_csv('geno_d2575.txt',header=False)
geno_d2575

### d2080 discretization

In [ ]:
geno_d20 = (temp_geno + 0.80).astype('int32')
geno_d80 = (temp_geno + 0.20).astype('int32')

geno_d2080 = geno_d20 + geno_d80

geno_d2080.insert(0,'!alle_1','G')
geno_d2080.insert(1,'!alle_2','C')
geno_d2080.to_csv('geno_d2080.txt',header=False)
geno_d2080

### d1585 discretization

In [ ]:
geno_d15 = (temp_geno + 0.85).astype('int32')
geno_d85 = (temp_geno + 0.15).astype('int32')

geno_d1585 = geno_d15 + geno_d85

geno_d1585.insert(0,'!alle_1','G')
geno_d1585.insert(1,'!alle_2','C')
geno_d1585.to_csv('geno_d1585.txt',header=False)
geno_d1585

### d1090 discretization

In [ ]:
geno_d10 = (temp_geno + 0.90).astype('int32')
geno_d90 = (temp_geno + 0.10).astype('int32')

geno_d1090 = geno_d10 + geno_d90

geno_d1090.insert(0,'!alle_1','G')
geno_d1090.insert(1,'!alle_2','C')
geno_d1090.to_csv('geno_d1090.txt',header=False)
geno_d1090

## Generating BIMBAM Phenotype File

get "pheno.txt" and "pheno_samples_in_same_order.txt". The latter lists the column names for the former.

In [ ]:
GSEmtx['!trans_age'].to_csv('pheno_horvathtrans.txt',index=False,header=False)


## EWAS by GEMMA

### EWAS of Continuous

In [ ]:
# compute Kinship matrix
!gemma -g geno_2x.txt -p pheno_horvathtrans.txt -gk -o kinship_2x.cxx.txt
# run univariate LMM
!gemma -g geno_2x.txt -p pheno_horvathtrans.txt -k kinship_2x.cXX.txt -lmm -o lmm_output_2x

### DEWAS of D2575

In [ ]:
# compute Kinship matrix
!gemma -g geno_d2575.txt -p pheno_horvathtrans.txt -gk -o kinship_d2575.cxx.txt
# run univariate LMM
!gemma -g geno_d2575.txt -p pheno_horvathtrans.txt -k kinship_d2575.cXX.txt -lmm -o lmm_output_d2575

### DEWAS of D2080

In [ ]:
# compute Kinship matrix
!gemma -g geno_d2080.txt -p pheno_horvathtrans.txt -gk -o kinship_d2080.cxx.txt
# run univariate LMM
!gemma -g geno_d2080.txt -p pheno_horvathtrans.txt -k kinship_d2080.cXX.txt -lmm -o lmm_output_d2080

### DEWAS of D1585

In [ ]:
# compute Kinship matrix
!gemma -g geno_d1585.txt -p pheno_horvathtrans.txt -gk -o kinship_d1585.cxx.txt
# run univariate LMM
!gemma -g geno_d1585.txt -p pheno_horvathtrans.txt -k kinship_d1585.cXX.txt -lmm -o lmm_output_d1585

### DEWAS of D1090

In [ ]:
# compute Kinship matrix
!gemma -g geno_d1090.txt -p pheno_horvathtrans.txt -gk -o kinship_d1090.cxx.txt
# run univariate LMM
!gemma -g geno_d1090.txt -p pheno_horvathtrans.txt -k kinship_d1090.cXX.txt -lmm -o lmm_output_d1090